# S AI LLM 1.0 · предобучение RU/EN

Запускайте ячейки по одной в среде Tesla T4. Короткий пилот проверяет обновление 20 Transformer-блоков, расход памяти и возобновление.

По умолчанию ноутбук работает только с выбранными файлами. Загрузите исходный `s_ai_llm_1_0_untrained_expanded.pt` через панель **Файлы** в `/content`. Для ускорения подготовки данных туда же можно загрузить `sai1b_wikipedia_pilot_10m.tar.gz`. В этом режиме скачайте новую папку результатов до завершения сессии. Если вам нужно автоматически читать и сохранять файлы в Google Drive, укажите `USE_DRIVE = True` и самостоятельно подтвердите запрос Colab. Это разрешение даёт коду ноутбука доступ к файлам диска.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys, tarfile

USE_DRIVE = False
disk = Path("/content/sai-1b")
disk.mkdir(exist_ok=True)
base_name = "s_ai_llm_1_0_untrained_expanded.pt"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive/S_AI_Training")
    reports = root / "S_AI_LLM_1_0" / "pretraining_v3_20261108"
    source = root / "outputs/s_ai_llm_1_0_expansion" / base_name
    old_source = root / "outputs/gpu_dialogue_v6/best.pt"
    v2_source = root / "S_AI_LLM_1_0/training_v2/best_output_head.pt"
    trial_source = root / "S_AI_LLM_1_0/output_head_trial_20_steps.pt"
else:
    reports = Path("/content/sai1b-results")
    source = Path("/content") / base_name
    old_source = Path("/content/source-v6.pt")
    v2_source = Path("/content/best_output_head.pt")
    trial_source = Path("/content/output_head_trial_20_steps.pt")

reports.mkdir(parents=True, exist_ok=True)
repo = disk / "S-AI-LLM"
branch = "codex/sai-llm-1b-pretraining"

def update_repo():
    if not repo.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", branch,
                        "https://github.com/thepeshkov7/S-AI-LLM.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", branch], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], check=True)

def run(module, *args):
    subprocess.run([sys.executable, "-u", "-m", "training_1b." + module, *map(str, args)],
                   cwd=repo, check=True)

def stage(source, name=None):
    if not source.is_file():
        raise FileNotFoundError(f"Загрузите или укажите файл {source}")
    if not USE_DRIVE:
        return source
    target = disk / (name or source.name)
    incoming = source.stat()
    if (not target.is_file() or target.stat().st_size != incoming.st_size
            or target.stat().st_mtime_ns != incoming.st_mtime_ns):
        temporary = target.with_suffix(target.suffix + ".partial")
        shutil.copy2(source, temporary)
        temporary.replace(target)
    return target

subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"], check=True)
update_repo()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers==0.21.4",
                "filelock", "pyarrow==20.0.0", "bitsandbytes==0.48.2"], check=True)
base = stage(source)
old = stage(old_source, "source-v6.pt") if old_source.is_file() else None
v2 = stage(v2_source) if v2_source.is_file() else None
trial = stage(trial_source) if trial_source.is_file() else None
output = reports / "t4-pilot"
configuration = repo / "training_1b/t4_pilot.json"

In [ ]:
# 1. Реальные ответы исходной модели. Дополнительный аудит, если доступны старые веса.
if old and not (reports / "checkpoint-audit.json").exists():
    heads = [arg for head in (v2, trial) if head for arg in ("--head", head)]
    run("checkpoint", "--base", base, "--previous", old, "--output",
        reports / "checkpoint-audit.json", *heads)
for label, head in (("expanded", None), ("training-v2", v2)):
    if label == "training-v2" and not head:
        continue
    target = reports / (label + "-answers.json")
    if not target.exists():
        run("evaluate", "--checkpoint", base, "--output", target,
            *(("--head", head) if head else ()))

In [ ]:
# 2. Пилотный корпус Wikipedia RU/EN с отдельными validation и test.
corpus = disk / "wiki-pilot"
archive = (reports if USE_DRIVE else Path("/content")) / "sai1b_wikipedia_pilot_10m.tar.gz"
if not corpus.exists() and archive.is_file():
    with tarfile.open(archive) as package:
        package.extractall(disk, filter="data")
if not corpus.exists():
    run("corpus", "--checkpoint", base, "--output", corpus,
        "--cache", disk / "dataset-cache", "--wiki-shards", 2, "--limit-tokens", 5_000_000)
for name in ("manifest.json", "source-manifest.json"):
    shutil.copyfile(corpus / name, reports / ("wiki-" + name))

In [ ]:
# 3. Первый отрезок. Обновляются все 20 Transformer-блоков.
# Если состояние уже существует, переходите к ячейке 4.
run("pretrain", "--profile", configuration, "--corpus", corpus, "--init", base,
    "--output", output, "--stop-after", 12)

In [ ]:
# 4. Возобновление после шага 12. При новом сеансе сначала выполните настройку и подготовку корпуса.
state = stage(output / "training-state.pt", "resume-state.pt")
run("pretrain", "--profile", configuration, "--corpus", corpus, "--resume", state,
    "--output", output)

In [ ]:
# 5. Независимые ответы после пилота.
trained = stage(output / "training-state.pt", "trained-state.pt")
if not (reports / "t4-pilot-answers.json").exists():
    run("evaluate", "--checkpoint", trained, "--output", reports / "t4-pilot-answers.json")
print("Результаты сохранены в", reports)